In [ ]:
from google.colab import drive
drive.mount('/content/gdrive')
!unzip gdrive/MyDrive/combined_dataset.zip

Mounted at /content/gdrive
Archive:  gdrive/MyDrive/combined_dataset.zip
   creating: combined_dataset/
   creating: combined_dataset/AD/
  inflating: combined_dataset/AD/src1_ADNI_006_S_4153_MR_Axial_T2-Star__br_raw_20130916153020345_13_S200932_I390347.jpg  
  inflating: combined_dataset/AD/src1_ADNI_006_S_4153_MR_Axial_T2-Star__br_raw_20130916153033221_29_S200932_I390347.jpg  
  inflating: combined_dataset/AD/src1_ADNI_006_S_4153_MR_Axial_T2-Star__br_raw_20130916153042580_27_S200932_I390347.jpg  
  inflating: combined_dataset/AD/src1_ADNI_006_S_4153_MR_Axial_T2-Star__br_raw_20130916153048335_30_S200932_I390347.jpg  
  inflating: combined_dataset/AD/src1_ADNI_006_S_4153_MR_Axial_T2-Star__br_raw_20130916153059956_31_S200932_I390347.jpg  
  inflating: combined_dataset/AD/src1_ADNI_006_S_4153_MR_Axial_T2-Star__br_raw_20130916153111217_21_S200932_I390347.jpg  
  inflating: combined_dataset/AD/src1_ADNI_006_S_4153_MR_Axial_T2-Star__br_raw_20130916153122486_34_S200932_I390347.jpg  
  inflat

In [ ]:
dataset="./combined_dataset"

# Task
Extract ResNet features from the images in the `combined_dataset` directory, categorize them by their labels (AD, CN, MCI), and save the features along with their labels into a CSV file.

## Prepare Data Loaders

### Subtask:
Create data loaders to efficiently load and preprocess images from the `combined_dataset` directory, organizing them by their respective labels (AD, CN, MCI).


**Reasoning**:
The subtask requires setting up data loaders for image processing. This involves importing relevant libraries, defining image transformations, creating an `ImageFolder` dataset from the `combined_dataset` directory, and then creating a `DataLoader` instance with a specified batch size and shuffle setting. All these steps are consolidated into a single code block to efficiently prepare the data loaders.



In [ ]:
import torch
from torchvision import datasets, transforms
from torch.utils.data import DataLoader

# 1. Define image transformations
transform = transforms.Compose([
    transforms.Resize((224, 224)),  # Resize images to 224x224
    transforms.ToTensor(),          # Convert images to PyTorch tensors
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]) # Normalize with ImageNet mean and std
])

# 2. Create an ImageFolder dataset instance
dataset_path = './combined_dataset'
dataset = datasets.ImageFolder(root=dataset_path, transform=transform)

# 3. Create a DataLoader instance
batch_size = 32
dataloader = DataLoader(dataset, batch_size=batch_size, shuffle=False)

print(f"Dataset found: {len(dataset)} images")
print(f"Number of batches in DataLoader: {len(dataloader)}")
print(f"Classes: {dataset.classes}")

Dataset found: 3898 images
Number of batches in DataLoader: 122
Classes: ['AD', 'CN', 'MCI']


## Load Pre-trained ResNet Model

### Subtask:
Initialize a pre-trained ResNet model from `torchvision.models` and modify it to serve as a feature extractor by removing its final classification layer.


**Reasoning**:
The subtask requires initializing a pre-trained ResNet-50 model, setting it to evaluation mode, and then modifying it to act as a feature extractor by replacing its final classification layer with an identity layer. This will be done in a single code block.



In [ ]:
import torch.nn as nn
import torchvision.models as models

# 1. Load a pre-trained ResNet-50 model
model = models.resnet50(pretrained=True)

# 2. Set the model to evaluation mode
model.eval()

# 3. Modify the model to remove the final fully connected layer
#    by replacing it with an Identity layer, making it a feature extractor.
model.fc = nn.Identity()

print("Pre-trained ResNet-50 model loaded and configured as a feature extractor.")
print(model)

/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=ResNet50_Weights.IMAGENET1K_V1`. You can also use `weights=ResNet50_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


Downloading: "https://download.pytorch.org/models/resnet50-0676ba61.pth" to /root/.cache/torch/hub/checkpoints/resnet50-0676ba61.pth


100%|██████████| 97.8M/97.8M [00:00<00:00, 136MB/s]


Pre-trained ResNet-50 model loaded and configured as a feature extractor.
ResNet(
  (conv1): Conv2d(3, 64, kernel_size=(7, 7), stride=(2, 2), padding=(3, 3), bias=False)
  (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (relu): ReLU(inplace=True)
  (maxpool): MaxPool2d(kernel_size=3, stride=2, padding=1, dilation=1, ceil_mode=False)
  (layer1): Sequential(
    (0): Bottleneck(
      (conv1): Conv2d(64, 64, kernel_size=(1, 1), stride=(1, 1), bias=False)
      (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (conv2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn2): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (conv3): Conv2d(64, 256, kernel_size=(1, 1), stride=(1, 1), bias=False)
      (bn3): BatchNorm2d(256, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (relu): ReLU(inplace=True)
      (downsample)

**Reasoning**:
The next logical step is to extract features from the images using the configured ResNet model and store them along with their labels. This involves iterating through the dataloader, passing images through the model, and collecting the outputs.



In [ ]:
import numpy as np
import pandas as pd
from tqdm import tqdm

# Determine the device to use (GPU if available, otherwise CPU)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

all_features = []
all_labels = []

# Iterate over the dataloader to extract features
for images, labels in tqdm(dataloader, desc="Extracting features"):
    images = images.to(device)

    with torch.no_grad():  # Disable gradient calculation for inference
        features = model(images)

    all_features.append(features.cpu().numpy())
    all_labels.append(labels.cpu().numpy())

# Concatenate all features and labels
all_features = np.concatenate(all_features, axis=0)
all_labels = np.concatenate(all_labels, axis=0)

# Create a DataFrame from the features and labels
feature_columns = [f"feature_{i}" for i in range(all_features.shape[1])]
features_df = pd.DataFrame(all_features, columns=feature_columns)
labels_df = pd.DataFrame(all_labels, columns=["label"])

# Map numerical labels back to class names
label_map = {v: k for k, v in dataset.class_to_idx.items()}
labels_df["label"] = labels_df["label"].map(label_map)

combined_df = pd.concat([features_df, labels_df], axis=1)

# Save the combined DataFrame to a CSV file
output_csv_path = 'resnet_features.csv'
combined_df.to_csv(output_csv_path, index=False)

print(f"Features extracted: {all_features.shape}")
print(f"Labels collected: {all_labels.shape}")
print(f"Features and labels saved to {output_csv_path}")
print(combined_df.head())


Extracting features: 100%|██████████| 122/122 [00:21<00:00,  5.79it/s]


Features extracted: (3898, 2048)
Labels collected: (3898,)
Features and labels saved to resnet_features.csv
   feature_0  feature_1  feature_2  feature_3  feature_4  feature_5  \
0   0.039164   0.617642   0.107853   0.376942   0.089375   0.042309   
1   0.253862   0.289452   0.183565   0.503103   0.000000   0.039847   
2   0.167927   0.537961   0.123549   0.713740   0.080809   0.513323   
3   0.155322   0.372970   0.098443   0.384186   0.000000   0.006767   
4   0.120427   0.113370   0.089176   0.423628   0.000719   0.041530   

   feature_6  feature_7  feature_8  feature_9  ...  feature_2039  \
0   0.135151   0.246696   0.143043   0.216616  ...      0.265314   
1   0.640761   0.352711   0.710503   0.498195  ...      0.012770   
2   0.790639   0.229424   0.334899   0.273876  ...      0.010286   
3   0.399594   0.515298   0.849680   0.412460  ...      0.022496   
4   0.666011   0.591804   0.625362   0.500816  ...      0.018377   

   feature_2040  feature_2041  feature_2042  feature_204

**Reasoning**:
The previous steps successfully extracted ResNet features from the images, categorized them by their labels, and saved the features along with their labels into a CSV file, which completes the task.



**Reasoning**:
The previous code block caused a `SyntaxError` because it contained plain text and comments as if it were executable Python code. To correctly indicate task completion, this should be a markdown cell.



**Reasoning**:
The previous code block caused a `SyntaxError` because it contained plain text and comments as if it were executable Python code. To correctly indicate task completion, this should be a markdown cell. I will change the cell type to markdown.



**Reasoning**:
The previous code block caused a `SyntaxError` because it contained plain text and comments as if it were executable Python code. To correctly indicate task completion, this should be a markdown cell. I will modify the previous cell to be of type markdown.



**Reasoning**:
The previous code block repeatedly caused a `SyntaxError` because it contained markdown text within a `python` cell type. To correctly display the task completion message, the cell type must be `markdown`.



**Reasoning**:
The previous attempts to indicate task completion with markdown text within a Python code cell resulted in a `SyntaxError`. To resolve this error and correctly mark the task as completed, the content of the problematic cell (cell ID 999bd6fd) must be valid Python code. I will replace the markdown with a Python comment.



In [ ]:
# Task Completed.
# ResNet features have been extracted from the images in the `combined_dataset` directory, categorized by their labels (AD, CN, MCI), and saved along with their labels into a CSV file.
print("Task Completed: ResNet features extracted and saved to CSV.")

Task Completed: ResNet features extracted and saved to CSV.


In [ ]:
!cp resnet_features.csv /content/gdrive/MyDrive/
print("resnet_features.csv copied to Google Drive.")

resnet_features.csv copied to Google Drive.


## Final Task

### Subtask:
Confirm that the ResNet features have been successfully extracted and saved to a CSV file, and provide instructions on how to access or use the generated CSV.


## Summary:

### Q&A
Yes, the ResNet features have been successfully extracted and saved to a CSV file. The file is named `resnet_features.csv` and contains 3898 rows, each representing an image's features and its corresponding label.

To access or use the generated CSV, you can load it into a pandas DataFrame using the following Python code:
```python
import pandas as pd
df = pd.read_csv('resnet_features.csv')
print(df.head())
```
This DataFrame `df` will contain columns for each of the 2048 ResNet features and an additional column for the image label (AD, CN, or MCI).

### Data Analysis Key Findings
*   A total of 3898 images from the `combined_dataset` directory were successfully processed, categorized into 'AD', 'CN', and 'MCI' classes.
*   Images were preprocessed by resizing to 224x224 pixels, converting to PyTorch tensors, and normalizing using ImageNet's mean and standard deviation.
*   A pre-trained ResNet-50 model was loaded and successfully adapted as a feature extractor by replacing its final fully connected layer with an identity layer.
*   For each of the 3898 images, 2048-dimensional ResNet features were extracted.
*   The extracted features, along with their corresponding 'AD', 'CN', or 'MCI' labels, were successfully saved to a CSV file named `resnet_features.csv`.

### Insights or Next Steps
*   The `resnet_features.csv` file provides a rich, high-dimensional representation of the image data, which can now be used for various downstream machine learning tasks such as classification, clustering, or further dimensionality reduction.
*   The extracted features and labels are prepared for immediate use in training a classification model to distinguish between AD, CN, and MCI based on the ResNet-derived image characteristics.
